**question-1**

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv('bestsellers with categories.csv')
ratings_series = pd.Series(df['User Rating'].values, index=df['Name'])
ratings_series.value_counts()


,count
4.8,127
4.7,108
4.6,105
4.5,60
4.9,52
4.4,38
4.3,25
4.0,14
4.2,8
4.1,6


question-2


In [23]:
prices = df['Price'].values
reviews = df['Reviews'].values
ratings = df['User Rating'].values
corr_price_reviews = np.corrcoef(prices, reviews)[0, 1]
corr_price_ratings = np.corrcoef(prices, ratings)[0, 1]
median_reviews = np.median(reviews)
high_reviews_mask = reviews > median_reviews
low_reviews_mask = reviews <= median_reviews
avg_price_high_reviews = np.mean(prices[high_reviews_mask])
avg_price_low_reviews = np.mean(prices[low_reviews_mask])
print("Corr Price-Reviews:", round(corr_price_reviews, 4))
print("Corr Price-Ratings:", round(corr_price_ratings, 4))
print("Avg Price High Reviews:", round(avg_price_high_reviews, 2))
print("Avg Price Low Reviews:", round(avg_price_low_reviews, 2))


Corr Price-Reviews: -0.1092
Corr Price-Ratings: -0.1331
Avg Price High Reviews: 10.99
Avg Price Low Reviews: 15.08


questi0n -3


In [24]:
df_hierarchical = df.set_index(['Genre', 'Year'])
grouped_means = df_hierarchical[['Price', 'User Rating']].groupby(['Genre', 'Year']).mean()
print(grouped_means.xs('Non Fiction', level='Genre'))

          Price  User Rating
Year                        
2009  15.230769     4.576923
2010  16.000000     4.520000
2011  17.620690     4.513793
2012  17.482759     4.558621
2013  18.192308     4.561538
2014  20.809524     4.609524
2015  10.969697     4.645455
2016  13.516129     4.654839
2017  13.730769     4.588462
2018  11.793103     4.617241
2019  10.566667     4.686667


question-4


In [25]:
max_reviews = df['Reviews'].max()
max_rating = df['User Rating'].max()

df['Pop_Index'] = (df['Reviews'] / max_reviews) + (df['User Rating'] / max_rating)
df['Rank'] = df['Pop_Index'].rank(ascending=False)

leaderboard = df.groupby('Author')['Pop_Index'].mean().reset_index()
leaderboard.sort_values(by='Pop_Index', ascending=False)


,Author,Pop_Index
68,Delia Owens,1.979592
180,Paula Hawkins,1.741164
168,Michelle Obama,1.675542
142,Kristin Hannah,1.540697
100,Gillian Flynn,1.468311
...,...,...
159,Mark Twain,0.862733
109,Ian K. Smith M.D.,0.862600
186,Pierre Dukan,0.859765
43,Chris Cleave,0.853435


question-5

In [26]:
df_missing = df.copy()
missing_idx = df_missing.sample(frac=0.1, random_state=42).index
df_missing.loc[missing_idx, 'Price'] = np.nan

df_missing['Price_Mean'] = df_missing['Price'].fillna(df_missing['Price'].mean())
df_missing['Price_Median'] = df_missing['Price'].fillna(df_missing['Price'].median())
df_missing['Price_Ffill'] = df_missing['Price'].fillna(method='ffill')

df_missing.groupby('Year')[['Price', 'Price_Mean', 'Price_Median', 'Price_Ffill']].mean()


/tmp/ipykernel_8967/1820945327.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_missing['Price_Ffill'] = df_missing['Price'].fillna(method='ffill')


,Price,Price_Mean,Price_Median,Price_Ffill
Year,,,,
2009,15.510638,15.358909,15.24,15.30
2010,13.804348,13.738545,13.58,13.82
2011,14.854167,14.779273,14.70,14.94
2012,15.652174,15.438545,15.28,15.46
2013,14.808511,14.698909,14.58,14.48
2014,13.025641,13.016000,12.58,13.82
2015,10.422222,10.678182,10.48,9.90
2016,12.404255,12.438909,12.32,12.80
2017,11.622222,11.758182,11.56,11.36


question-6

In [27]:
high_books = df[(df['User Rating'] >= 4.8) & (df['Reviews'] > 10000)]
high_books['Genre'].value_counts()


,count
Genre,
Fiction,59
Non Fiction,26


In [28]:

high_books['Author'].value_counts()

,count
Author,
Dr. Seuss,8
Eric Carle,7
Sarah Young,6
Giles Andreae,5
Harper Lee,5
Gary Chapman,5
Laura Hillenbrand,5
R. J. Palacio,5
Kathryn Stockett,4


question-7

In [29]:
def price_tier(price):
    if price < 10:
        return 'Budget'
    elif price <= 20:
        return 'Standard'
    else:
        return 'Premium'

df['Tier'] = df['Price'].apply(price_tier)
df.groupby(['Year', 'Tier'])['Name'].count()


Year  Tier    
2009  Budget      18
      Premium      7
      Standard    25
2010  Budget      14
      Premium      5
      Standard    31
2011  Budget      14
      Premium     11
      Standard    25
2012  Budget      14
      Premium     11
      Standard    25
2013  Budget      18
      Premium      5
      Standard    27
2014  Budget      24
      Premium      9
      Standard    17
2015  Budget      29
      Premium      2
      Standard    19
2016  Budget      26
      Premium      7
      Standard    17
2017  Budget      29
      Premium      7
      Standard    14
2018  Budget      28
      Premium      3
      Standard    19
2019  Budget      24
      Premium      1
      Standard    25
Name: Name, dtype: int64



> **question-8e**



In [30]:
genre_avg = df.groupby('Genre')['Price'].mean()

df['Price_Dev'] = df['Price'] - df['Genre'].map(genre_avg)

overpriced = df[df['Price_Dev'] > 0]
underpriced = df[df['Price_Dev'] < 0]

# Show overpriced books
overpriced[['Name', 'Price_Dev']]


,Name,Price_Dev
1,11/22/63: A Novel,11.150000
2,12 Rules for Life: An Antidote to Chaos,0.158065
5,A Dance with Dragons (A Song of Ice and Fire),0.150000
6,A Game of Thrones / A Clash of Kings / A Storm...,19.150000
7,A Gentleman in Moscow: A Novel,4.150000
...,...,...
529,What Should Danny Do? (The Power to Choose Ser...,2.150000
534,Where the Crawdads Sing,4.150000
535,Where the Wild Things Are,2.150000
537,Wild: From Lost to Found on the Pacific Crest ...,3.158065


question-9

In [31]:
fict = df[df['Genre'] == 'Fiction'].mean(numeric_only=True)
non_fict = df[df['Genre'] == 'Non Fiction'].mean(numeric_only=True)

fict[['User Rating', 'Price', 'Reviews']] - non_fict[['User Rating', 'Price', 'Reviews']]


,0
User Rating,0.053172
Price,-3.991935
Reviews,6618.646505


question-10

In [32]:
final_df = df.copy()
final_df['Price'] = final_df['Price'].fillna(final_df['Price'].mean())

df_multi = final_df.set_index(['Genre', 'Year'])
df_multi['Pop_Score'] = np.multiply(df_multi['User Rating'], df_multi['Reviews'])

df_multi.groupby(['Genre', 'Year'])[['Price', 'Pop_Score']].mean()


Price      Pop_Score
Genre       Year                          
Fiction     2009  15.583333   30094.808333
            2010   9.700000   39028.920000
            2011  11.619048   48060.338095
            2012  12.285714   87806.042857
            2013  10.708333   88404.929167
            2014  10.172414   88576.968966
            2015   9.352941  108430.247059
            2016  12.631579   89982.500000
            2017   8.833333   68610.304167
            2018   8.761905   60243.909524
            2019   9.350000   88817.455000
Non Fiction 2009  15.230769   13787.750000
            2010  16.000000   16239.463333
            2011  17.620690   29930.982759
            2012  17.482759   37568.868966
            2013  18.192308   30904.630769
            2014  20.809524   51672.928571
            2015  10.969697   43756.815152
            2016  13.516129   51043.680645
            2017  13.730769   52208.326923
            2018  11.793103   69016.741379
            2019  10.566667   66512.040000